# Regression: Model without PCA vs Model with PCA (paired t-test)

## Imports

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, time, warnings
from scipy import stats
from sklearn.model_selection import train_test_split, KFold, GridSearchCV, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import get_scorer
warnings.filterwarnings("ignore")

## Config

In [ ]:
DATA = "regression_data.csv"
TARGET = "Remaining_Useful_Life_Hours"
ID_COL = "Record_ID"
SEED = 42
ALPHA = 0.05

## Load Data

In [ ]:
df = pd.read_csv(DATA)
df.head()

## Shape

In [ ]:
print(df.shape)

## Data Types

In [ ]:
df.info()

## Missing Values

In [ ]:
miss = df.isna().sum()
miss[miss > 0]

## Duplicate Rows

In [ ]:
print(df.duplicated().sum())

## Descriptive Statistics

In [ ]:
df.describe().T

## Identify Identifier Column

In [ ]:
df.nunique().sort_values(ascending=False).head(3)

## Feature Types

In [ ]:
X_cols = df.drop(columns=[TARGET, ID_COL])
num_cols = X_cols.select_dtypes(include="number").columns.tolist()
cat_cols = X_cols.select_dtypes(exclude="number").columns.tolist()
print("numeric:", num_cols)
print("categorical:", cat_cols)

## Skewness

In [ ]:
df[num_cols + [TARGET]].skew().sort_values()

## Target Distribution (Histogram and QQ Plot)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df[TARGET], kde=True, ax=ax[0])
stats.probplot(df[TARGET], plot=ax[1])
plt.show()

## Predictor Histograms

In [ ]:
df[num_cols].hist(bins=30, figsize=(14, 9))
plt.tight_layout()
plt.show()

## Predictor QQ Plots

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for col, ax in zip(num_cols, axes.ravel()):
    stats.probplot(df[col].dropna(), plot=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Categorical Counts

In [ ]:
fig, axes = plt.subplots(1, len(cat_cols), figsize=(10, 3.5))
for col, ax in zip(cat_cols, np.atleast_1d(axes)):
    sns.countplot(x=df[col], ax=ax)
plt.show()

## Correlation Heatmap

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(df[num_cols + [TARGET]].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.show()

## Highly Correlated Pairs

In [ ]:
corr = df[num_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
pairs = upper.stack()
pairs[pairs > 0.8].sort_values(ascending=False)

## Outlier Boxplots

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 8))
for col, ax in zip(num_cols, axes.ravel()):
    sns.boxplot(y=df[col], ax=ax)
plt.tight_layout()
plt.show()

## Outlier Count (IQR Rule)

In [ ]:
Q1, Q3 = df[num_cols].quantile(0.25), df[num_cols].quantile(0.75)
IQR = Q3 - Q1
outliers = (df[num_cols] < Q1 - 1.5 * IQR) | (df[num_cols] > Q3 + 1.5 * IQR)
outliers.sum()

## Drop Identifier

In [ ]:
df = df.drop(columns=ID_COL)

## Impute Categorical (fillna)

In [ ]:
df[cat_cols] = df[cat_cols].fillna("Unknown")
df[cat_cols].isna().sum()

## Encode Categorical

In [ ]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
df.shape

## Train Test Split

In [ ]:
X = df.drop(columns=TARGET)
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train_raw, X_test_raw = X_train.copy(), X_test.copy()
print(X_train.shape, X_test.shape)

## Cross-Validation Folds (same for both models)

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

## Impute Numeric (fillna with train median)

In [ ]:
med = X_train[num_cols].median()
X_train = X_train.fillna(med)
X_test = X_test.fillna(med)
print(X_train.isna().sum().sum(), X_test.isna().sum().sum())

## Outlier Treatment (IQR clip, bounds from train)

In [ ]:
Q1, Q3 = X_train[num_cols].quantile(0.25), X_train[num_cols].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
X_train[num_cols] = X_train[num_cols].clip(lower, upper, axis=1)
X_test[num_cols] = X_test[num_cols].clip(lower, upper, axis=1)

## Scale Features

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

## Scoring Metrics

In [ ]:
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
    "mape": "neg_mean_absolute_percentage_error",
}

## Model Choice
Run exactly ONE of the model cells below. Run All uses the last one (SVR).

### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
param_grid = {"fit_intercept": [True, False]}

### Ridge

In [ ]:
from sklearn.linear_model import Ridge
model = Ridge()
param_grid = {"alpha": [0.01, 0.1, 1, 10, 100]}

### Lasso

In [ ]:
from sklearn.linear_model import Lasso
model = Lasso(max_iter=10000)
param_grid = {"alpha": [0.001, 0.01, 0.1, 1]}

### ElasticNet

In [ ]:
from sklearn.linear_model import ElasticNet
model = ElasticNet(max_iter=10000)
param_grid = {"alpha": [0.001, 0.01, 0.1, 1], "l1_ratio": [0.2, 0.5, 0.8]}

### KNN Regressor

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
model = KNeighborsRegressor()
param_grid = {"n_neighbors": [3, 5, 7, 9, 11, 15], "weights": ["uniform", "distance"], "p": [1, 2]}

### Decision Tree Regressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor
model = DecisionTreeRegressor(random_state=SEED)
param_grid = {"max_depth": [3, 5, 10, None], "min_samples_leaf": [1, 5, 10], "min_samples_split": [2, 10]}

### Random Forest Regressor

In [ ]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=SEED)
param_grid = {"n_estimators": [100, 200], "max_depth": [5, 10, None], "min_samples_leaf": [1, 3, 5]}

### Gradient Boosting Regressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
model = GradientBoostingRegressor(random_state=SEED)
param_grid = {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [2, 3, 4]}

### AdaBoost Regressor

In [ ]:
from sklearn.ensemble import AdaBoostRegressor
model = AdaBoostRegressor(random_state=SEED)
param_grid = {"n_estimators": [50, 100, 200], "learning_rate": [0.05, 0.1, 1]}

### XGBoost Regressor

In [ ]:
from xgboost import XGBRegressor
model = XGBRegressor(random_state=SEED)
param_grid = {"n_estimators": [100, 300], "max_depth": [3, 5], "learning_rate": [0.05, 0.1]}

### SVR (RBF kernel)

In [ ]:
from sklearn.svm import SVR
model = SVR(kernel="rbf")
param_grid = {"C": [1, 10, 100, 1000], "gamma": ["scale", 0.01, 0.1], "epsilon": [0.1, 1, 10]}

## Tune Without PCA

In [ ]:
grid1 = GridSearchCV(model, param_grid, cv=cv, scoring=scoring["rmse"], n_jobs=-1)
grid1.fit(X_train_s, y_train)

## Best Parameters Without PCA

In [ ]:
print(grid1.best_params_)
print("CV RMSE:", -grid1.best_score_)

## Cross-Validate Without PCA

In [ ]:
res1 = cross_validate(grid1.best_estimator_, X_train_s, y_train, cv=cv, scoring=scoring,
                      return_train_score=True, n_jobs=-1)

## PCA: Components for 95% Variance

In [ ]:
pca_full = PCA().fit(X_train_s)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_comp = int(np.argmax(cumvar >= 0.95) + 1)
print("components for 95%:", n_comp)

## PCA: Explained Variance Plot

In [ ]:
plt.plot(range(1, len(cumvar) + 1), cumvar, marker="o")
plt.axhline(0.95, color="r", linestyle="--")
plt.axvline(n_comp, color="g", linestyle="--")
plt.xlabel("Components")
plt.ylabel("Cumulative explained variance")
plt.show()

## PCA: Explained Variance Table

In [ ]:
pd.DataFrame({
    "PC": range(1, n_comp + 1),
    "Explained": pca_full.explained_variance_ratio_[:n_comp],
    "Cumulative": cumvar[:n_comp],
}).round(4)

## Apply PCA

In [ ]:
pca = PCA(n_components=n_comp).fit(X_train_s)
X_train_pca = pca.transform(X_train_s)
X_test_pca = pca.transform(X_test_s)
print("original dims:", X_train_s.shape[1], "| reduced dims:", X_train_pca.shape[1])

## Tune With PCA

In [ ]:
grid2 = GridSearchCV(model, param_grid, cv=cv, scoring=scoring["rmse"], n_jobs=-1)
grid2.fit(X_train_pca, y_train)

## Best Parameters With PCA

In [ ]:
print(grid2.best_params_)
print("CV RMSE:", -grid2.best_score_)

## Cross-Validate With PCA

In [ ]:
res2 = cross_validate(grid2.best_estimator_, X_train_pca, y_train, cv=cv, scoring=scoring,
                      return_train_score=True, n_jobs=-1)

## Fold-wise RMSE Table

In [ ]:
rmse1 = -res1["test_rmse"]
rmse2 = -res2["test_rmse"]
fold_table = pd.DataFrame({
    "Fold": range(1, len(rmse1) + 1),
    "RMSE Without PCA": rmse1,
    "RMSE With PCA": rmse2,
    "Difference (With - Without)": rmse2 - rmse1,
}).round(3)
fold_table

## Mean and Std of All Metrics

In [ ]:
def val(res, k, split="test"):
    v = res[f"{split}_{k}"]
    return -v if scoring[k].startswith("neg_") else v

rows = {}
for k in scoring:
    rows[k.upper()] = [val(res1, k).mean(), val(res1, k).std(), val(res2, k).mean(), val(res2, k).std()]
pd.DataFrame(rows, index=["Mean (no PCA)", "Std (no PCA)", "Mean (PCA)", "Std (PCA)"]).T.round(4)

## Train vs Validation Gap

In [ ]:
gap = {}
for name, res in [("Without PCA", res1), ("With PCA", res2)]:
    tr, va = val(res, "rmse", "train").mean(), val(res, "rmse", "test").mean()
    gap[name] = {"Train RMSE": tr, "Validation RMSE": va, "Gap (Val - Train)": va - tr}
pd.DataFrame(gap).T.round(3)

## Paired t-test (two-sided)

In [ ]:
d = rmse2 - rmse1
t_stat, p_val = stats.ttest_rel(rmse2, rmse1)
p_val_main = p_val
print("mean difference:", d.mean().round(4))
print("t =", round(t_stat, 4), "| p =", round(p_val, 4))
print("Reject H0" if p_val < ALPHA else "Fail to reject H0", f"at alpha = {ALPHA}")

## Fit on Train and Evaluate on Test

In [ ]:
t0 = time.time(); m1 = grid1.best_estimator_.fit(X_train_s, y_train); fit1 = time.time() - t0
t0 = time.time(); m2 = grid2.best_estimator_.fit(X_train_pca, y_train); fit2 = time.time() - t0
test = {}
for k, s in scoring.items():
    sign = -1 if s.startswith("neg_") else 1
    test[k.upper()] = [sign * get_scorer(s)(m1, X_test_s, y_test), sign * get_scorer(s)(m2, X_test_pca, y_test)]
pd.DataFrame(test, index=["Without PCA", "With PCA"]).T.round(4)

## Predicted vs Actual

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, pred, name in zip(ax, [m1.predict(X_test_s), m2.predict(X_test_pca)], ["Without PCA", "With PCA"]):
    a.scatter(y_test, pred, alpha=0.5)
    a.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
    a.set_title(name); a.set_xlabel("Actual"); a.set_ylabel("Predicted")
plt.show()

## Residual Plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, pred, name in zip(ax, [m1.predict(X_test_s), m2.predict(X_test_pca)], ["Without PCA", "With PCA"]):
    a.scatter(pred, y_test - pred, alpha=0.5)
    a.axhline(0, color="r", linestyle="--")
    a.set_title(name); a.set_xlabel("Predicted"); a.set_ylabel("Residual")
plt.show()

## Final Comparison Table

In [ ]:
cmp = {"Dimensions": [X_train_s.shape[1], X_train_pca.shape[1]]}
for k in scoring:
    cmp["CV " + k.upper()] = [val(res1, k).mean(), val(res2, k).mean()]
cmp["Val - Train RMSE gap"] = [val(res1, "rmse").mean() - val(res1, "rmse", "train").mean(),
                               val(res2, "rmse").mean() - val(res2, "rmse", "train").mean()]
cmp["Fit time (s)"] = [fit1, fit2]
cmp["t-test p-value"] = [p_val_main, p_val_main]
cmp["Best params"] = [str(grid1.best_params_), str(grid2.best_params_)]
pd.DataFrame(cmp, index=["Without PCA", "With PCA"]).T

## Final Answer Points (write in your own words)
- Preprocessing, dimensionality, same folds, metrics, t-test result, generalization gap, cost, interpretability, reproducibility (seed)
- Limitations: one 80/20 split, small data, PCA components are hard to interpret, only a grid search
- Improvements: nested CV, log-transform of the skewed target, wider or randomized search, try other kernels or models

## Pipeline Version (everything inside the training folds)

In [ ]:
pipe1 = Pipeline([("imputer", SimpleImputer(strategy="median")),
                  ("scaler", StandardScaler()),
                  ("model", model)])
pipe2 = Pipeline([("imputer", SimpleImputer(strategy="median")),
                  ("scaler", StandardScaler()),
                  ("pca", PCA(n_components=0.95)),
                  ("model", model)])
pipe_grid = {"model__" + k: v for k, v in param_grid.items()}

## Pipeline Tuning Inside Folds (nested CV)

In [ ]:
inner = KFold(n_splits=3, shuffle=True, random_state=SEED)
gs1 = GridSearchCV(pipe1, pipe_grid, cv=inner, scoring=scoring["rmse"], n_jobs=-1)
gs2 = GridSearchCV(pipe2, pipe_grid, cv=inner, scoring=scoring["rmse"], n_jobs=-1)
pres1 = cross_validate(gs1, X_train_raw, y_train, cv=cv, scoring=scoring, return_train_score=True)
pres2 = cross_validate(gs2, X_train_raw, y_train, cv=cv, scoring=scoring, return_train_score=True)

## Pipeline Fold-wise RMSE

In [ ]:
prmse1, prmse2 = -pres1["test_rmse"], -pres2["test_rmse"]
pd.DataFrame({"Fold": range(1, len(prmse1) + 1), "RMSE Without PCA": prmse1,
              "RMSE With PCA": prmse2, "Difference": prmse2 - prmse1}).round(3)

## Pipeline Paired t-test

In [ ]:
t_stat, p_val = stats.ttest_rel(prmse2, prmse1)
print("t =", round(t_stat, 4), "| p =", round(p_val, 4))
print("Reject H0" if p_val < ALPHA else "Fail to reject H0", f"at alpha = {ALPHA}")

## Pipeline Final Fit and Test Score

In [ ]:
final1 = gs1.fit(X_train_raw, y_train)
final2 = gs2.fit(X_train_raw, y_train)
print(final1.best_params_, "| test RMSE:", -get_scorer(scoring["rmse"])(final1, X_test_raw, y_test))
print(final2.best_params_, "| test RMSE:", -get_scorer(scoring["rmse"])(final2, X_test_raw, y_test))